# 01 — Data Collection and Cleaning

This notebook prepares the raw air-quality and meteorological datasets for analysis and machine-learning modeling.

The primary datasets are:

- UK-AIR AURN hourly air-quality measurements for selected London monitoring stations.
- UK Met Office MIDAS hourly weather observations from Heathrow.

The cleaning workflow includes:

1. Loading the raw datasets.
2. Extracting pollutant measurements from the AURN file.
3. Standardizing timestamps and variable names.
4. Removing invalid and duplicate observations.
5. Restricting the study period to 2021–2024.
6. Processing the annual Heathrow weather files.
7. Merging air-quality and weather observations by timestamp.
8. Validating and saving the processed datasets.

The final dataset will be used for exploratory analysis and PM2.5 prediction.

### 1. Import libraries and define project paths

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")

Project root: C:\Users\USER\Documents\WORK\uk_air_quality_project
Raw data: C:\Users\USER\Documents\WORK\uk_air_quality_project\data\raw
Processed data: C:\Users\USER\Documents\WORK\uk_air_quality_project\data\processed


### 2. Define the AURN source file

The AURN dataset downloaded from UK-AIR is stored in the `data/raw` directory.

Before loading the file, we verify that the expected file exists. This helps identify incorrect file paths early.

In [3]:
AURN_FILE = RAW_DIR / "94529965511.csv"

print(AURN_FILE)
print(f"Exists: {AURN_FILE.exists()}")

C:\Users\USER\Documents\WORK\uk_air_quality_project\data\raw\94529965511.csv
Exists: True


### 3. Load the raw AURN dataset

The UK-AIR download contains several metadata and explanatory rows before the actual tabular header.

Based on the structure of the downloaded file, the first 17 rows are skipped so that row 18 becomes the dataframe header.

`low_memory=False` is used because the file contains mixed data types across some columns.

In [4]:
aurn_raw = pd.read_csv(
    AURN_FILE,
    skiprows=17,
    low_memory=False
)

print("Shape:", aurn_raw.shape)
aurn_raw.head()

Shape: (43825, 88)


,Date,Time,Nitrogen dioxide,Status,PM10 particulate matter (Hourly measured),Status.1,PM2.5 particulate matter (Hourly measured),Status.2,Ozone,Status.3,Nitrogen dioxide.1,Status.4,PM10 particulate matter (Hourly measured).1,Status.5,PM2.5 particulate matter (Hourly measured).1,Status.6,Ozone.1,Status.7,Nitrogen dioxide.2,Status.8,PM10 particulate matter (Hourly measured).2,Status.9,PM2.5 particulate matter (Hourly measured).2,Status.10,Nitrogen dioxide.3,Status.11,PM2.5 particulate matter (Hourly measured).3,Status.12,Ozone.2,Status.13,Nitrogen dioxide.4,Status.14,PM10 particulate matter (Hourly measured).3,Status.15,PM2.5 particulate matter (Hourly measured).4,Status.16,Ozone.3,Status.17,Nitrogen dioxide.5,Status.18,PM10 particulate matter (Hourly measured).4,Status.19,PM2.5 particulate matter (Hourly measured).5,Status.20,Ozone.4,Status.21,Nitrogen dioxide.6,Status.22,PM10 particulate matter (Hourly measured).5,Status.23,PM2.5 particulate matter (Hourly measured).6,Status.24,Ozone.5,Status.25,PM10 particulate matter (Hourly measured).6,Status.26,PM2.5 particulate matter (Hourly measured).7,Status.27,Ozone.6,Status.28,Nitrogen dioxide.7,Status.29,PM10 particulate matter (Hourly measured).7,Status.30,PM2.5 particulate matter (Hourly measured).8,Status.31,Ozone.7,Status.32,Nitrogen dioxide.8,Status.33,PM10 particulate matter (Hourly measured).8,Status.34,PM2.5 particulate matter (Hourly measured).9,Status.35,PM10 particulate matter (Hourly measured).9,Status.36,PM2.5 particulate matter (Hourly measured).10,Status.37,PM10 particulate matter (Hourly measured).10,Status.38,PM2.5 particulate matter (Hourly measured).11,Status.39,Ozone.8,Status.40,Nitrogen dioxide.9,Status.41,PM2.5 particulate matter (Hourly measured).12,Status.42
0,2021-01-01,01:00:00,22.61433,V ugm-3,45.525,V ugm-3 (FIDAS),39.552,V ugm-3 (Ref.eq),29.53636,V ugm-3,28.35369,V ugm-3,18.358,V ugm-3 (Ref.eq),20,V ugm-3 (BAM),No data,V ugm-3,17.15938,V ugm-3,NaN,NaN,No data,V ugm-3 (TEOM FDMS),NaN,NaN,NaN,NaN,23.48274,V ugm-3,21.51916,V ugm-3,NaN,NaN,NaN,NaN,30.13507,V ugm-3,16.93883,V ugm-3,25.838,V ugm-3 (FIDAS),22.453,V ugm-3 (Ref.eq),27.80675,V ugm-3,17.06444,V ugm-3,NaN,NaN,NaN,NaN,NaN,NaN,33.275,V ugm-3 (FIDAS),29.316,V ugm-3 (Ref.eq),28.53851,V ugm-3,20.99867,V ugm-3,21.3,V ugm-3 (TEOM FDMS),18.6,V ugm-3 (TEOM FDMS),28.53851,V ugm-3,21.65101,V ugm-3,35.15,V ugm-3 (FIDAS),30.448,V ugm-3 (Ref.eq),NaN,NaN,NaN,NaN,35.275,V ugm-3 (FIDAS),31.085,V ugm-3 (Ref.eq),NaN,NaN,24.8457,V ugm-3,20,V ugm-3 (BAM)
1,2021-01-01,02:00:00,25.12337,V ugm-3,60.375,V ugm-3 (FIDAS),52.359,V ugm-3 (Ref.eq),34.92475,V ugm-3,25.03474,V ugm-3,22.223,V ugm-3 (Ref.eq),19,V ugm-3 (BAM),No data,V ugm-3,No data,V ugm-3,NaN,NaN,No data,V ugm-3 (TEOM FDMS),NaN,NaN,NaN,NaN,27.14152,V ugm-3,15.19117,V ugm-3,NaN,NaN,NaN,NaN,19.35829,V ugm-3,21.89902,V ugm-3,21.028,V ugm-3 (FIDAS),19.099,V ugm-3 (Ref.eq),21.62008,V ugm-3,19.14258,V ugm-3,NaN,NaN,NaN,NaN,NaN,NaN,21.625,V ugm-3 (FIDAS),19.293,V ugm-3 (Ref.eq),22.15227,V ugm-3,26.71965,V ugm-3,45.1,V ugm-3 (TEOM FDMS),40.3,V ugm-3 (TEOM FDMS),14.90123,V ugm-3,30.42826,V ugm-3,65.475,V ugm-3 (FIDAS),55.802,V ugm-3 (Ref.eq),NaN,NaN,NaN,NaN,32.175,V ugm-3 (FIDAS),28.585,V ugm-3 (Ref.eq),NaN,NaN,20.14728,V ugm-3,26,V ugm-3 (BAM)
2,2021-01-01,03:00:00,13.95652,V ugm-3,23.775,V ugm-3 (FIDAS),20.92,V ugm-3 (Ref.eq),38.86626,V ugm-3,22.63855,V ugm-3,22.223,V ugm-3 (Ref.eq),14,V ugm-3 (BAM),No data,V ugm-3,13.57075,V ugm-3,NaN,NaN,No data,V ugm-3 (TEOM FDMS),NaN,NaN,NaN,NaN,33.12862,V ugm-3,13.48811,V ugm-3,NaN,NaN,NaN,NaN,19.15872,V ugm-3,20.41308,V ugm-3,23.18,V ugm-3 (FIDAS),20.925,V ugm-3 (Ref.eq),19.25851,V ugm-3,19.22149,V ugm-3,NaN,NaN,NaN,NaN,NaN,NaN,18.775,V ugm-3 (FIDAS),16.604,V ugm-3 (Ref.eq),31.03314,V ugm-3,21.01699,V ugm-3,25.6,V ugm-3 (TEOM FDMS),23.9,V ugm-3 (TEOM FDMS),31.63185,V ugm-3,19.81852,V ugm-3,32.675,V ugm-3 (FIDAS),28.278,V ugm-3 (Ref.eq),NaN,NaN,NaN,NaN,27.175,V ugm-3 (FIDAS),24.481,V ugm-3 (Ref.eq),NaN,NaN,22.26946,V ugm-3,24,V ugm-3

### 4. Define the AURN monitoring stations

The downloaded dataset contains multiple London AURN monitoring sites arranged horizontally across the file.

The dictionary below records the starting column and name of each station identified from the AURN header.

Keeping this mapping explicitly documented makes the extraction process reproducible.

In [5]:
stations = {
    2: "London Bexley",
    8: "London Bloomsbury",
    16: "London Eltham",
    24: "London Farringdon Street",
    28: "London Haringey Priory Park South",
    36: "London Harlington",
    44: "London Hillingdon",
    52: "London Honor Oak Park",
    58: "London Marylebone Road",
    66: "London N. Kensington",
    74: "London Norbury Manor School",
    78: "London Teddington Bushy Park",
    82: "London Westminster"
}

print(f"Stations found: {len(stations)}")

Stations found: 13


### 5. Create a function for pollutant-column identification

Different AURN stations do not measure exactly the same pollutants.

For example, some stations contain NO2, PM10, PM2.5 and ozone, while others contain only a subset.

Rather than assuming that every station has the same column structure, this function searches each station's block for the requested pollutant.

In [6]:
def get_pollutant_column(df, station_start, pollutant):
    """
    Find the first column containing the requested pollutant
    within a station's column block.
    """

    station_end = min(station_start + 8, len(df.columns))

    for col_idx in range(station_start, station_end):
        column_name = str(df.columns[col_idx])

        if pollutant in column_name:
            return col_idx

    return None

### 6. Extract pollutant measurements for each station

We now convert the wide AURN format into a long, analysis-friendly format.

For every monitoring station, we extract:

- NO2 — nitrogen dioxide
- PM10 — particulate matter
- PM2.5 — fine particulate matter
- O3 — ozone

If a station does not measure a pollutant, its value is recorded as missing rather than incorrectly assigning another pollutant's values.

In [7]:
records = []

pollutants = {
    "NO2": "Nitrogen dioxide",
    "PM10": "PM10 particulate matter",
    "PM2.5": "PM2.5 particulate matter",
    "O3": "Ozone"
}

for station_start, station_name in stations.items():

    station_data = pd.DataFrame({
        "Date": aurn_raw["Date"],
        "Time": aurn_raw["Time"],
        "Station": station_name
    })

    for pollutant, search_name in pollutants.items():

        col_idx = get_pollutant_column(
            aurn_raw,
            station_start,
            search_name
        )

        if col_idx is not None:
            station_data[pollutant] = pd.to_numeric(
                aurn_raw.iloc[:, col_idx],
                errors="coerce"
            )
        else:
            station_data[pollutant] = np.nan

    records.append(station_data)

aurn_clean = pd.concat(
    records,
    ignore_index=True
)

print("Shape:", aurn_clean.shape)
aurn_clean.head()

Shape: (569725, 7)


,Date,Time,Station,NO2,PM10,PM2.5,O3
0,2021-01-01,01:00:00,London Bexley,22.61433,45.525,39.552,29.53636
1,2021-01-01,02:00:00,London Bexley,25.12337,60.375,52.359,34.92475
2,2021-01-01,03:00:00,London Bexley,13.95652,23.775,20.920,38.86626
3,2021-01-01,04:00:00,London Bexley,12.19039,17.750,15.943,22.55141
4,2021-01-01,05:00:00,London Bexley,13.42547,19.125,17.076,10.07828


### 7. Combine date and time into a datetime variable

The original AURN dataset stores date and time separately.

For time-series analysis, forecasting and merging with meteorological data, we need one standardized timestamp.

The two columns are therefore combined into a single `Datetime` column.

In [8]:
aurn_clean["Datetime"] = pd.to_datetime(
    aurn_clean["Date"].astype(str) + " " +
    aurn_clean["Time"].astype(str),
    errors="coerce"
)

aurn_clean = aurn_clean.drop(
    columns=["Date", "Time"]
)

aurn_clean = aurn_clean[
    ["Datetime", "Station", "NO2", "PM10", "PM2.5", "O3"]
]

print(aurn_clean.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569725 entries, 0 to 569724
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   Datetime  545974 non-null  datetime64[ns]
 1   Station   569725 non-null  object        
 2   NO2       402019 non-null  float64       
 3   PM10      350570 non-null  float64       
 4   PM2.5     373533 non-null  float64       
 5   O3        399197 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 26.1+ MB
None


### 8. Remove invalid timestamps and duplicate observations

Rows without a valid timestamp cannot be used for time-series modeling.

We also remove duplicate observations for the same station and timestamp to ensure that each station contributes at most one observation per hour.

In [9]:
before = len(aurn_clean)

aurn_clean = aurn_clean.dropna(
    subset=["Datetime"]
)

aurn_clean = aurn_clean.drop_duplicates(
    subset=["Datetime", "Station"]
)

after = len(aurn_clean)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {after:,}")
print(f"Removed:              {before - after:,}")

Rows before cleaning: 569,725
Rows after cleaning:  545,974
Removed:              23,751


### 9. Restrict the dataset to the study period

The project uses 2021–2024 as the common study period.

This period is important because the available AURN and Heathrow MIDAS weather datasets overlap across these four complete years.

Restricting both datasets to the same period prevents accidental use of observations outside the modeling window.

In [10]:
START_DATE = "2021-01-01 00:00:00"
END_DATE = "2024-12-31 23:00:00"

aurn_clean = aurn_clean[
    (aurn_clean["Datetime"] >= START_DATE) &
    (aurn_clean["Datetime"] <= END_DATE)
].copy()

print("Start:", aurn_clean["Datetime"].min())
print("End:", aurn_clean["Datetime"].max())
print("Shape:", aurn_clean.shape)

Start: 2021-01-01 01:00:00
End: 2024-12-31 23:00:00
Shape: (436839, 6)


In [22]:
# Checking cleaned pollutant values

print(
    aurn_clean[["NO2", "PM10", "PM2.5", "O3"]].min()
)

NO2      0.0
PM10     0.0
PM2.5    0.0
O3       0.0
dtype: float64


### 10. Select stations with sufficient PM2.5 coverage

The initial AURN dataset contains 13 London stations, but their PM2.5 data availability varies considerably.

For the first modeling phase, we retain the five stations with substantially better PM2.5 coverage:

- London Harlington
- London N. Kensington
- London Honor Oak Park
- London Marylebone Road
- London Bloomsbury

This reduces the amount of missing target data while retaining spatial diversity across London.

## Station Selection
Five AURN stations with the highest data completeness were selected to improve data quality and reduce missing observations in the modelling dataset.

In [25]:
# Selecting high-quality AURN stations

SELECTED_STATIONS = [
    "London Harlington",
    "London N. Kensington",
    "London Honor Oak Park",
    "London Marylebone Road",
    "London Bloomsbury"
]

aurn_model = aurn_clean[
    aurn_clean["Station"].isin(SELECTED_STATIONS)
].copy()

print(aurn_model["Station"].value_counts())

Station
London Bloomsbury         33603
London Harlington         33603
London Honor Oak Park     33603
London Marylebone Road    33603
London N. Kensington      33603
Name: count, dtype: int64


In [26]:
# Checking selected AURN data

print(
    aurn_model[["NO2", "PM10", "PM2.5", "O3"]].min()
)

NO2      0.0
PM10     0.0
PM2.5    0.0
O3       0.0
dtype: float64


### 11. Create a weather-data processing function

The Heathrow MIDAS weather data are supplied as separate annual files.

This function standardizes each file into the variables required for the air-quality model:

- Temperature
- Dewpoint
- Humidity
- Wind speed
- Wind direction
- Pressure
- Visibility

Creating a reusable function allows the same processing logic to be applied consistently to 2021–2024.

In [12]:
def process_weather_file(file_path):
    """
    Load and standardize a MIDAS Heathrow hourly weather file.
    """

    weather = pd.read_csv(
        file_path,
        skiprows=283,
        na_values="NA"
    )

    weather = weather[
        [
            "ob_time",
            "air_temperature",
            "dewpoint",
            "rltv_hum",
            "wind_speed",
            "wind_direction",
            "msl_pressure",
            "visibility"
        ]
    ].copy()

    weather = weather.rename(columns={
        "ob_time": "Datetime",
        "air_temperature": "Temperature",
        "dewpoint": "Dewpoint",
        "rltv_hum": "Humidity",
        "wind_speed": "WindSpeed",
        "wind_direction": "WindDirection",
        "msl_pressure": "Pressure",
        "visibility": "Visibility"
    })

    weather["Datetime"] = pd.to_datetime(
        weather["Datetime"],
        errors="coerce"
    )

    return weather

### 12. Define the annual weather files

The weather data were downloaded separately for each year.

We store the four raw-file paths in a dictionary so that they can be processed automatically rather than writing separate code for every year.

In [13]:
WEATHER_FILES = {
    2021: RAW_DIR / "heathrow_2021.csv",
    2022: RAW_DIR / "heathrow_2022.csv",
    2023: RAW_DIR / "heathrow_2023.csv",
    2024: RAW_DIR / "heathrow_2024.csv"
}

for year, path in WEATHER_FILES.items():
    print(year, "→", path.exists())

2021 → True
2022 → True
2023 → True
2024 → True


### 13. Process and combine the annual weather datasets

Each annual weather file is processed using the function defined above.

The resulting datasets are then concatenated into one continuous 2021–2024 weather dataset.

In [14]:
weather_records = []

for year, file_path in WEATHER_FILES.items():

    print(f"Processing {year}...")

    yearly_weather = process_weather_file(file_path)

    yearly_weather["Year"] = year

    weather_records.append(yearly_weather)

weather_all = pd.concat(
    weather_records,
    ignore_index=True
)

print("Weather shape:", weather_all.shape)

Processing 2021...
Processing 2022...


C:\Users\USER\AppData\Local\Temp\ipykernel_1716\3843258645.py:6: DtypeWarning: Columns (94,95) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.read_csv(


Processing 2023...


C:\Users\USER\AppData\Local\Temp\ipykernel_1716\3843258645.py:6: DtypeWarning: Columns (94) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.read_csv(


Processing 2024...
Weather shape: (35066, 9)


C:\Users\USER\AppData\Local\Temp\ipykernel_1716\3843258645.py:6: DtypeWarning: Columns (94) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.read_csv(


### 14. Clean the weather dataset

Invalid timestamps and duplicate hourly observations are removed.

The weather dataset is also restricted to the same 2021–2024 study period used for the AURN observations.

In [15]:
weather_all = weather_all.dropna(
    subset=["Datetime"]
)

weather_all = weather_all.drop_duplicates(
    subset=["Datetime"]
)

weather_all = weather_all[
    (weather_all["Datetime"] >= START_DATE) &
    (weather_all["Datetime"] <= END_DATE)
].copy()

print("Weather shape:", weather_all.shape)
print("Start:", weather_all["Datetime"].min())
print("End:", weather_all["Datetime"].max())

Weather shape: (35062, 9)
Start: 2021-01-01 00:00:00
End: 2024-12-31 23:00:00


### 15. Merge air-quality and meteorological observations

The AURN and weather datasets are both hourly.

We therefore merge them using the `Datetime` column.

A left join is used so that every valid AURN air-quality observation is retained, even when a weather observation is temporarily unavailable.

In [29]:
weather_for_merge = weather_all.drop(
    columns=["Year"],
    errors="ignore"
)

merged_df = aurn_model.merge(
    weather_for_merge,
    on="Datetime",
    how="left"
)

print("Final merged shape:", merged_df.shape)
merged_df.head()

Final merged shape: (168015, 13)


,Datetime,Station,NO2,PM10,PM2.5,O3,Temperature,Dewpoint,Humidity,WindSpeed,WindDirection,Pressure,Visibility
0,2021-01-01 01:00:00,London Bloomsbury,28.35369,18.358,20.0,29.53636,0.2,-0.7,91.8,2.0,340.0,1009.6,350.0
1,2021-01-01 02:00:00,London Bloomsbury,25.03474,22.223,19.0,34.92475,0.2,-0.6,93.9,2.0,320.0,1009.9,400.0
2,2021-01-01 03:00:00,London Bloomsbury,22.63855,22.223,14.0,38.86626,0.0,-0.6,95.7,4.0,280.0,1010.3,320.0
3,2021-01-01 04:00:00,London Bloomsbury,52.45771,17.392,17.0,22.55141,-0.2,-0.7,95.7,4.0,290.0,1010.3,250.0
4,2021-01-01 05:00:00,London Bloomsbury,75.17713,24.155,20.0,10.07828,-0.7,-1.0,97.2,6.0,270.0,1010.2,50.0


### 16. Validate the merged dataset

Before saving the data, we check:

- The overall date range.
- The number of observations per station.
- The percentage of missing values.

This validation step helps identify problems introduced during extraction, cleaning or merging.

In [17]:
print("Date range:")
print(merged_df["Datetime"].min())
print(merged_df["Datetime"].max())

print("\nStations:")
print(merged_df["Station"].value_counts())

print("\nMissing values (%):")
print(
    merged_df.isna()
    .mean()
    .mul(100)
    .round(2)
)

Date range:
2021-01-01 01:00:00
2024-12-31 23:00:00

Stations:
Station
London Bloomsbury         33603
London Harlington         33603
London Honor Oak Park     33603
London Marylebone Road    33603
London N. Kensington      33603
Name: count, dtype: int64

Missing values (%):
Datetime          0.00
Station           0.00
NO2              22.47
PM10              3.43
PM2.5             8.13
O3               20.99
Temperature       0.02
Dewpoint          0.02
Humidity          0.03
WindSpeed         1.67
WindDirection     1.67
Pressure          0.03
Visibility        0.55
dtype: float64


In [30]:
# Final AURN quality check

print(
    aurn_model[["NO2", "PM10", "PM2.5", "O3"]].min()
)

NO2      0.0
PM10     0.0
PM2.5    0.0
O3       0.0
dtype: float64


### 17. Save the processed datasets

The cleaned AURN data, cleaned weather data and final merged dataset are saved in `data/processed`.

Keeping raw and processed data separate ensures that the original downloaded files remain untouched and reproducible.

In [31]:
aurn_output = PROCESSED_DIR / "london_aurn_pm25.csv"
weather_output = PROCESSED_DIR / "heathrow_weather_2021_2024.csv"
merged_output = PROCESSED_DIR / "london_air_quality_weather_2021_2024.csv"

aurn_model.to_csv(
    aurn_output,
    index=False
)

weather_all.to_csv(
    weather_output,
    index=False
)

merged_df.to_csv(
    merged_output,
    index=False
)

print("Saved:")
print(aurn_output)
print(weather_output)
print(merged_output)

Saved:
C:\Users\USER\Documents\WORK\uk_air_quality_project\data\processed\london_aurn_pm25.csv
C:\Users\USER\Documents\WORK\uk_air_quality_project\data\processed\heathrow_weather_2021_2024.csv
C:\Users\USER\Documents\WORK\uk_air_quality_project\data\processed\london_air_quality_weather_2021_2024.csv


In [32]:
# Checking saved datasets

aurn_check = pd.read_csv(aurn_output)
merged_check = pd.read_csv(merged_output)

print("AURN minimums:")
print(
    aurn_check[["NO2", "PM10", "PM2.5", "O3"]].min()
)

print("\nMerged minimums:")
print(
    merged_check[["NO2", "PM10", "PM2.5", "O3"]].min()
)

AURN minimums:
NO2      0.0
PM10     0.0
PM2.5    0.0
O3       0.0
dtype: float64

Merged minimums:
NO2      0.0
PM10     0.0
PM2.5    0.0
O3       0.0
dtype: float64


### 18. Final quality-control summary

This final check provides a concise summary of the dataset that will be passed to the exploratory-analysis stage.

We verify the number of rows, columns, date coverage, variables and missing-data percentages.

In [19]:
print("=" * 60)
print("FINAL DATASET QUALITY CONTROL")
print("=" * 60)

print(f"Rows:    {len(merged_df):,}")
print(f"Columns: {len(merged_df.columns)}")

print(f"\nStart date: {merged_df['Datetime'].min()}")
print(f"End date:   {merged_df['Datetime'].max()}")

print("\nColumns:")
for column in merged_df.columns:
    print(f"  - {column}")

print("\nMissing values (%):")
print(
    merged_df.isna()
    .mean()
    .mul(100)
    .round(2)
)

print("\nStation counts:")
print(merged_df["Station"].value_counts())

print("\nDataset saved successfully.")

FINAL DATASET QUALITY CONTROL
Rows:    168,015
Columns: 13

Start date: 2021-01-01 01:00:00
End date:   2024-12-31 23:00:00

Columns:
  - Datetime
  - Station
  - NO2
  - PM10
  - PM2.5
  - O3
  - Temperature
  - Dewpoint
  - Humidity
  - WindSpeed
  - WindDirection
  - Pressure
  - Visibility

Missing values (%):
Datetime          0.00
Station           0.00
NO2              22.47
PM10              3.43
PM2.5             8.13
O3               20.99
Temperature       0.02
Dewpoint          0.02
Humidity          0.03
WindSpeed         1.67
WindDirection     1.67
Pressure          0.03
Visibility        0.55
dtype: float64

Station counts:
Station
London Bloomsbury         33603
London Harlington         33603
London Honor Oak Park     33603
London Marylebone Road    33603
London N. Kensington      33603
Name: count, dtype: int64

Dataset saved successfully.


In [20]:
from pathlib import Path
import pandas as pd

processed_dir = Path("../data/processed")

files = {
    "AURN": processed_dir / "london_aurn_pm25.csv",
    "Weather": processed_dir / "heathrow_weather_2021_2024.csv",
    "Merged": processed_dir / "london_air_quality_weather_2021_2024.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Shape:", df.shape)
    print("Date range:", df["Datetime"].min(), "→", df["Datetime"].max())
    print("Columns:", df.columns.tolist())

    print("\nMissing values (%):")
    print(
        df.isna()
        .mean()
        .mul(100)
        .round(2)
    )


AURN
Shape: (168015, 6)
Date range: 2021-01-01 01:00:00 → 2024-12-31 23:00:00
Columns: ['Datetime', 'Station', 'NO2', 'PM10', 'PM2.5', 'O3']

Missing values (%):
Datetime     0.00
Station      0.00
NO2         22.47
PM10         3.43
PM2.5        8.13
O3          20.99
dtype: float64

Weather
Shape: (35062, 9)
Date range: 2021-01-01 00:00:00 → 2024-12-31 23:00:00
Columns: ['Datetime', 'Temperature', 'Dewpoint', 'Humidity', 'WindSpeed', 'WindDirection', 'Pressure', 'Visibility', 'Year']

Missing values (%):
Datetime         0.00
Temperature      0.02
Dewpoint         0.02
Humidity         0.03
WindSpeed        1.65
WindDirection    1.65
Pressure         0.02
Visibility       0.56
Year             0.00
dtype: float64

Merged
Shape: (168015, 13)
Date range: 2021-01-01 01:00:00 → 2024-12-31 23:00:00
Columns: ['Datetime', 'Station', 'NO2', 'PM10', 'PM2.5', 'O3', 'Temperature', 'Dewpoint', 'Humidity', 'WindSpeed', 'WindDirection', 'Pressure', 'Visibility']

Missing values (%):
Datetime     

In [21]:
# Remove invalid pollutant values

pollutants = ["NO2", "PM10", "PM2.5", "O3"]

for pollutant in pollutants:
    aurn_clean.loc[
        aurn_clean[pollutant] < 0,
        pollutant
    ] = np.nan

print("Negative pollutant values after cleaning:")

print(
    (aurn_clean[pollutants] < 0)
    .sum()
)

Negative pollutant values after cleaning:
NO2      0
PM10     0
PM2.5    0
O3       0
dtype: int64


In [28]:
# Verifying saved AURN dataset

aurn_check = pd.read_csv(aurn_output)

print(
    aurn_check[["NO2", "PM10", "PM2.5", "O3"]].min()
)

NO2      0.0
PM10     0.0
PM2.5    0.0
O3       0.0
dtype: float64
